# 🚀 Qwen-Image-2.1 Uncensored (Q4_0 GGUF) Colab 프로비저닝 & 서빙

이 노트북은 [abenzerps/Qwen-Image-2.1-Uncensored-GGUF](https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF) 모델을 **Google Colab에서 ComfyUI와 Cloudflare Tunnel을 통해 즉시 웹 UI로 서빙**할 수 있도록 제작되었습니다.

---
### 📌 주요 구성 파일
1. **Diffusion Model**: `qwen-image-2.1-UC-Q4_0.gguf` (약 4.15 GB)
2. **Text Encoder**: `qwen3vl_8b_int8_convrot.safetensors` (약 9.35 GB, 저메모리 최적화)
3. **VAE**: `qwen_image_2.1_vae_bf16.safetensors` (약 676 MB)
4. **ComfyUI-GGUF** (leejet 포크: Qwen-Image 2.1 네이티브 지원)
5. **Cloudflare Tunnel**: 외부 브라우저 접속을 위한 터널링 (`--enable-cors-header` 및 `--listen 0.0.0.0` 적용)

👉 **상단 메뉴에서 `런타임 > 모두 실행(Run all)`**을 클릭하시면 모든 과정이 전자동으로 진행됩니다.

### 1단계: GPU 환경 확인
Colab 상단 메뉴 `런타임 > 런타임 유형 변경`에서 **T4 GPU** 이상으로 설정되어 있는지 확인합니다.

In [ ]:
!nvidia-smi

### 2단계: 필수 도구 및 ComfyUI 설치
- 초고속 다중 다운로더(`aria2`) 및 외부 접속 터널링(`cloudflared`) 설치
- 최신 ComfyUI 및 Qwen-Image-2.1을 지원하는 `ComfyUI-GGUF` 커스텀 노드 설치

In [ ]:
# 1. aria2 및 cloudflared 설치
!apt-get update -qq && apt-get install -y -qq aria2
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# 2. ComfyUI 클론 및 기본 패키지 설치
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!pip install -q -r requirements.txt
!pip install -q gguf huggingface_hub

# 3. Qwen-Image-2.1을 네이티브 지원하는 leejet의 ComfyUI-GGUF 설치
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/leejet/ComfyUI-GGUF.git
%cd /content/ComfyUI/custom_nodes/ComfyUI-GGUF
!pip install -q -r requirements.txt
%cd /content/ComfyUI
print('✅ ComfyUI 및 확장 노드 설치 완료!')

### 3단계: Qwen-Image-2.1 Uncensored 모델 파일 다운로드
`aria2c`를 사용하여 Hugging Face에서 3가지 핵심 모델 파일을 고속 다운로드합니다.
(403 Forbidden 방지를 위해 브라우저 User-Agent 헤더를 포함합니다)

In [ ]:
import os

# 필요한 폴더 경로 생성
os.makedirs('/content/ComfyUI/models/diffusion_models', exist_ok=True)
os.makedirs('/content/ComfyUI/models/text_encoders', exist_ok=True)
os.makedirs('/content/ComfyUI/models/vae', exist_ok=True)

ua = '--user-agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"'

# 1. Diffusion Model (Q4_0 GGUF)
print('📥 [1/3] Diffusion Model: qwen-image-2.1-UC-Q4_0.gguf 다운로드 중 (약 4.15 GB)...')
!aria2c --console-log-level=error -c -x 8 -s 8 -k 1M $ua \
  'https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF/resolve/main/qwen-image-2.1-UC-Q4_0.gguf' \
  -d /content/ComfyUI/models/diffusion_models -o qwen-image-2.1-UC-Q4_0.gguf

# 2. Text Encoder (INT8 ConvRot - 메모리 최적화)
print('📥 [2/3] Text Encoder: qwen3vl_8b_int8_convrot.safetensors 다운로드 중 (약 9.35 GB)...')
!aria2c --console-log-level=error -c -x 8 -s 8 -k 1M $ua \
  'https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors' \
  -d /content/ComfyUI/models/text_encoders -o qwen3vl_8b_int8_convrot.safetensors

# 3. VAE (BF16)
print('📥 [3/3] VAE: qwen_image_2.1_vae_bf16.safetensors 다운로드 중 (약 676 MB)...')
!aria2c --console-log-level=error -c -x 8 -s 8 -k 1M $ua \
  'https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors' \
  -d /content/ComfyUI/models/vae -o qwen_image_2.1_vae_bf16.safetensors

print('✅ 모든 모델 파일 다운로드 완료!')

### 4단계: Cloudflare Tunnel & ComfyUI 실행 (외부 접속 서빙)
- **403 Forbidden 방지 설정**:
  - `--listen 0.0.0.0`: 모든 인터페이스 접속 허용
  - `--enable-cors-header`: 외부 터널 도메인에서의 웹 UI 및 API 요청 허용
  - `--lowvram`: Colab T4 GPU VRAM 한도 초과 방지
- 터널이 연결되면 공개 URL(`https://*.trycloudflare.com`)이 콘솔에 출력됩니다.

In [ ]:
import subprocess
import threading
import time
import re

%cd /content/ComfyUI

# 1. Cloudflare Tunnel 실행 (포트 8188)
tunnel_cmd = ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188']
tunnel_proc = subprocess.Popen(tunnel_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

def monitor_tunnel():
    for line in iter(tunnel_proc.stderr.readline, ''):
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            url = match.group(0)
            print('\n' + '='*65)
            print(f'🎉 ComfyUI 서빙 URL: {url}')
            print('👉 위 링크를 새 브라우저 창에서 열면 바로 ComfyUI가 실행됩니다!')
            print('='*65 + '\n')
            break

threading.Thread(target=monitor_tunnel, daemon=True).start()
time.sleep(3)

# 2. ComfyUI 시작 (--listen 0.0.0.0 과 --enable-cors-header 로 403 Forbidden 차단 방지)
!python main.py --listen 0.0.0.0 --port 8188 --enable-cors-header --lowvram --preview-method auto

### 💡 ComfyUI 웹 UI 설정 및 생성 방법
위 링크로 ComfyUI 화면에 접속하신 후 다음 순서대로 노드를 연결합니다:

1. **Diffusion Model 노드**:
   - 빈 화면 우클릭 > `Add Node` > `advanced/loaders` > **`Unet Loader (GGUF)`** 추가
   - `unet_name`에 `qwen-image-2.1-UC-Q4_0.gguf` 선택

2. **Text Encoder (CLIP) 노드**:
   - 빈 화면 우클릭 > `Add Node` > `loaders` > **`CLIPLoader`** 추가
   - `clip_name`에 `qwen3vl_8b_int8_convrot.safetensors` 선택

3. **VAE 노드**:
   - 빈 화면 우클릭 > `Add Node` > `loaders` > **`VAELoader`** 추가
   - `vae_name`에 `qwen_image_2.1_vae_bf16.safetensors` 선택

4. **샘플러 & 출력 연결**:
   - `CLIP Text Encode (Prompt)`에 긍정/부정 프롬프트 작성
   - `KSampler` 연결 (Steps: 20~30, CFG: 3.5~5.0 권장)
   - `Empty Latent Image` (해상도 1024x1024 권장) 연결
   - `VAE Decode` -> `Save Image` 연결
   - 우측 메뉴의 **`Queue Prompt`** 버튼을 누르면 이미지 생성이 시작됩니다!